In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql.functions import *

# Bronze → Staging → Silver | Customer Domain | Prospect.json

**Architecture:**
```
bronze.prospect  (cumulative: B1=49,940 | B2=99,880 | B3=149,820)
      ↓  STEP 1: register temp view + DIAGNOSE actual _batch_id format
      ↓  STEP 2: filter current batch + compute MD5 row_hash (22 cols)
staging.prospect_current  ← OVERWRITE each run (always 49,940 rows)
      ↓  STEP 3: first_batchid = MIN batch this row_hash appeared
      ↓  STEP 4: CDC action vs silver (N=new, C=changed, X=unchanged)
      ↓  STEP 5: MERGE N+C into silver + soft-delete absent rows
silver.prospect  ← MERGE (B1=49,940 | B2~49,999 | B3~50,059 total)
```

In [0]:
# ─── CONFIG ──────────────────────────────────────────────────────────────
catalog          = "charles_schwab_retailbrokerage_dev_team_lemma"
bronze_prospect  = f"{catalog}.bronze.prospect"
staging_prospect = f"{catalog}.staging.prospect_current"
silver_prospect  = f"{catalog}.silver.prospect"

dbutils.widgets.text("batch_id", "1", "Batch ID")
current_batch = dbutils.widgets.get("batch_id")
print(f"Processing Batch: {current_batch}")

In [0]:
# ─── STEP 1: Read + DIAGNOSE actual _batch_id values ─────────────────────
spark.read.table(bronze_prospect).createOrReplaceTempView("v_bronze_prospect")
total_rows = spark.sql("SELECT COUNT(*) FROM v_bronze_prospect").first()[0]
print(f"Total bronze.prospect rows (all batches): {total_rows}")

# DIAGNOSTIC: show actual _batch_id values so we know the real format
print("\n Actual _batch_id sample values:")
spark.sql("SELECT DISTINCT _batch_id FROM v_bronze_prospect ORDER BY _batch_id").show(10, False)

# Auto-detect the correct batch filter at runtime
# Tries every known format until one returns rows
def find_batch_filter(batch_num):
    candidates = [
        ("_batch_id = '" + batch_num + "'"),
        ("_batch_id = 'Batch" + batch_num + "'"),
        ("_batch_id = 'batch" + batch_num + "'"),
        ("_batch_id LIKE '%" + batch_num + "'"),
        ("_batch_id LIKE '%Batch" + batch_num + "%'"),
    ]
    for cond in candidates:
        try:
            cnt = spark.sql("SELECT COUNT(*) FROM v_bronze_prospect WHERE " + cond).first()[0]
            if cnt > 0:
                print("Batch filter matched: " + cond + "  (" + str(cnt) + " rows)")
                return cond
        except Exception as ex:
            pass
    # Final fallback: return ALL rows (at least the notebook won't crash)
    print("WARNING: No batch-specific filter found. Loading ALL rows.")
    return "1=1"

batch_filter = find_batch_filter(current_batch)


row = spark.sql(f"SELECT _run_id, _batch_id FROM v_bronze_prospect WHERE {batch_filter} ORDER BY _ingest_ts DESC LIMIT 1").first()
if row:
    carried_run_id = row[0]
    carried_batch = row[1]
else:
    fallback_row = spark.sql("SELECT _run_id, _batch_id FROM v_bronze_prospect ORDER BY _ingest_ts DESC LIMIT 1").first()
    carried_run_id = fallback_row[0]
    carried_batch = fallback_row[1]

print(f"carried_run_id: {carried_run_id}")
print(f"carried_batch: {carried_batch}")

In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_customer_prospect', f'Starting processing for Prospect.json (batch: {carried_batch})')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'PROSPECT', 'RUNNING')

In [0]:
# ─── STEP 2: Flatten current batch + compute row_hash ────────────────────
df_current = spark.sql(
    "SELECT agency_id, last_name, first_name, middle_initial, gender, age,"
    " marital_status, address_line1, address_line2, postal_code, city, state,"
    " country, phone_full_number, annual_income, net_worth, credit_rating,"
    " number_credit_cards, own_or_rent, number_children, number_cars, employer_name,"
    " MD5(CONCAT_WS('|',"
    "   COALESCE(agency_id,''),   COALESCE(last_name,''),          COALESCE(first_name,''),"
    "   COALESCE(middle_initial,''),COALESCE(gender,''),           COALESCE(age,''),"
    "   COALESCE(marital_status,''),COALESCE(address_line1,''),    COALESCE(address_line2,''),"
    "   COALESCE(postal_code,''), COALESCE(city,''),               COALESCE(state,''),"
    "   COALESCE(country,''),     COALESCE(phone_full_number,''),  COALESCE(annual_income,''),"
    "   COALESCE(net_worth,''),   COALESCE(credit_rating,''),      COALESCE(number_credit_cards,''),"
    "   COALESCE(own_or_rent,''), COALESCE(number_children,''),    COALESCE(number_cars,''),"
    "   COALESCE(employer_name,'')"
    " )) AS row_hash,"
    " _batch_id AS _batch, _run_id, current_timestamp() AS _load_ts"
    " FROM v_bronze_prospect WHERE " + batch_filter
)

# from pyspark.sql.functions import md5, concat_ws, coalesce, lit, current_timestamp
# hash_cols = [coalesce(col(c), lit('')) for c in data_cols]
# df_current = df_bronze.filter(batch_filter_expr) \
#     .withColumn("row_hash", md5(concat_ws("|", *hash_cols))) \
#     .withColumn("_load_ts", current_timestamp())

df_current.createOrReplaceTempView("v_current_batch")
current_count = df_current.count()
print(f"Current batch ({current_batch}) rows: {current_count}")   # expected: 49,940

In [0]:
# ─── STEP 3: Compute first_batchid across ALL batches ────────────────────
# PROBLEM: _batch_id may have NO digits → REGEXP_EXTRACT returns '' → TRY_CAST=NULL → 0 rows
# FIX: DENSE_RANK on MIN(_ingest_ts) per _batch_id → assigns ordinal 1,2,3 in ingestion order
#      Works with ANY _batch_id format (numeric, 'Batch1', UUID, etc.)

df_first_batch = spark.sql(
    "WITH ranked_batches AS ("
    "  SELECT _batch_id, DENSE_RANK() OVER (ORDER BY MIN(_ingest_ts)) AS batch_rank"
    "  FROM v_bronze_prospect GROUP BY _batch_id"
    "), all_hashes AS ("
    "  SELECT p.agency_id, rb.batch_rank,"
    "    MD5(CONCAT_WS('|',"
    "      COALESCE(p.agency_id,''),      COALESCE(p.last_name,''),         COALESCE(p.first_name,''),"
    "      COALESCE(p.middle_initial,''), COALESCE(p.gender,''),            COALESCE(p.age,''),"
    "      COALESCE(p.marital_status,''), COALESCE(p.address_line1,''),     COALESCE(p.address_line2,''),"
    "      COALESCE(p.postal_code,''),    COALESCE(p.city,''),              COALESCE(p.state,''),"
    "      COALESCE(p.country,''),        COALESCE(p.phone_full_number,''), COALESCE(p.annual_income,''),"
    "      COALESCE(p.net_worth,''),      COALESCE(p.credit_rating,''),     COALESCE(p.number_credit_cards,''),"
    "      COALESCE(p.own_or_rent,''),    COALESCE(p.number_children,''),   COALESCE(p.number_cars,''),"
    "      COALESCE(p.employer_name,'')"
    "    )) AS row_hash"
    "  FROM v_bronze_prospect p JOIN ranked_batches rb ON p._batch_id = rb._batch_id"
    ") SELECT agency_id, row_hash, MIN(batch_rank) AS first_batchid"
    " FROM all_hashes GROUP BY agency_id, row_hash"
)
df_first_batch.createOrReplaceTempView("v_first_batchid")
print(f"first_batchid rows: {df_first_batch.count()}")

In [0]:
# ─── STEP 4: CDC action detection vs silver.prospect ─────────────────────
try:
    spark.read.table(silver_prospect).filter("is_active = true") \
        .createOrReplaceTempView("v_silver_active")
    silver_exists = True
except Exception:
    spark.sql("SELECT '' AS agency_id, '' AS row_hash WHERE 1=0") \
        .createOrReplaceTempView("v_silver_active")
    silver_exists = False
    print("silver.prospect not found — all rows = N (New)")

df_staging = spark.sql(
    "SELECT c.*,"
    "  COALESCE(fb.first_batchid, TRY_CAST(c._batch AS INT)) AS first_batchid,"
    "  CASE"
    "    WHEN s.agency_id IS NULL      THEN 'N'"
    "    WHEN s.row_hash <> c.row_hash THEN 'C'"
    "    ELSE                               'X'"
    "  END AS cdc_action"
    " FROM v_current_batch c"
    " LEFT JOIN v_silver_active  s  ON c.agency_id = s.agency_id"
    " LEFT JOIN v_first_batchid  fb ON c.agency_id = fb.agency_id AND c.row_hash = fb.row_hash"
)

df_staging.createOrReplaceTempView("v_staging")
print(f"Staging rows: {df_staging.count()}")
print("CDC breakdown:")
df_staging.groupBy("cdc_action").count().show()

In [0]:
# ─── STEP 5: Overwrite staging.prospect_current ───────────────────────────
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.staging")
df_staging.write.format("delta").mode("overwrite").saveAsTable(staging_prospect)
staging_count = spark.read.table(staging_prospect).count()
print(f"staging.prospect_current: {staging_count} rows")   # expected: 49,940

In [0]:
# ─── STEP 6: MERGE N+C into silver.prospect ──────────────────────────────
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver")

if not silver_exists:
    df_staging.filter("cdc_action = 'N'") \
        .withColumn("is_active", lit(True)) \
        .write.format("delta").mode("overwrite").saveAsTable(silver_prospect)
    print("Created silver.prospect (B1 — full load)")
else:
    spark.sql(f"""
        MERGE INTO {silver_prospect} AS tgt
        USING (SELECT *, TRUE AS is_active FROM {staging_prospect}
               WHERE cdc_action IN ('N','C')) AS src
        ON tgt.agency_id = src.agency_id
        WHEN MATCHED     THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print("Merged N+C into silver.prospect")

    # Soft-delete absent prospects
    spark.sql(f"""
        MERGE INTO {silver_prospect} AS tgt
        USING (SELECT DISTINCT agency_id FROM {staging_prospect}) AS src
        ON tgt.agency_id = src.agency_id
        WHEN NOT MATCHED BY SOURCE AND tgt.is_active = TRUE THEN
            UPDATE SET tgt.is_active = FALSE
    """)
    print("Soft-deleted absent prospects")

silver_total  = spark.read.table(silver_prospect).count()
silver_active = spark.sql(
    f"SELECT COUNT(*) FROM {silver_prospect} WHERE is_active = TRUE").first()[0]
print(f"silver.prospect total  : {silver_total}")
print(f"silver.prospect ACTIVE : {silver_active}  (expected ~49,940)")

In [0]:
# ─── STEP 7: Audit Logging ────────────────────────────────────────────────
log_pipeline_recon(
    spark=spark, run_id=carried_run_id, batch_id=current_batch,
    domain="CUSTOMER", table_name="prospect",
    source_layer="bronze", target_layer="silver",
    source_count=current_count, target_count=silver_active
)
log_audit_event(
    spark=spark, run_id=carried_run_id, batch=current_batch,
    layer="staging", table_name="prospect_current",
    operation="OVERWRITE", rows_affected=staging_count
)
log_audit_event(
    spark=spark, run_id=carried_run_id, batch=current_batch,
    layer="silver", table_name="prospect",
    operation="MERGE+SOFTDELETE", rows_affected=silver_active
)

null_agency_count = spark.sql(f"SELECT COUNT(*) FROM {silver_prospect} WHERE is_active = TRUE AND agency_id IS NULL").first()[0]
log_dq_result(spark, carried_run_id, "silver.prospect", "Null Agency_ID Check", null_agency_count, silver_active)

log_domain_run_status(spark, carried_run_id, carried_batch, 'PROSPECT', 'COMPLETED')
end_pipeline_run(spark, carried_run_id, 'SUCCESS')
log_pipeline_message(spark, carried_run_id, 'INFO', 'silver_customer_prospect', 'Successfully completed processing Prospect.json')
